## Tokenization check

In [1]:
import re
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from transformers import AutoTokenizer

from src.data import load_ds, TEXT_COLUMNS
from src.preprocessing import clean_text
from project2.preprocessing import INPUT_FORMATS, encode_split, collate

ds = load_ds()
tok = AutoTokenizer.from_pretrained("roberta-base")
print("vocab size:", tok.vocab_size, "| max length:", tok.model_max_length)
print(tok.special_tokens_map)

vocab size: 50265 | max length: 512
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


In [2]:
# tokens before / after clean_text
for s in ["He didn’t   know what to do\nnext.", "It was Tom´s birthday.", "Broken \ufffd char.", "She didnt go home."]:
    print(repr(s))
    print("  raw  :", tok.tokenize(s))
    print("  clean:", tok.tokenize(clean_text(s)))

'He didn’t   know what to do\nnext.'
  raw  : ['He', 'Ġdidn', 'âĢ', 'Ļ', 't', 'Ġ', 'Ġ', 'Ġknow', 'Ġwhat', 'Ġto', 'Ġdo', 'Ċ', 'next', '.']
  clean: ['He', 'Ġdidn', "'t", 'Ġknow', 'Ġwhat', 'Ġto', 'Ġdo', 'Ġnext', '.']
'It was Tom´s birthday.'
  raw  : ['It', 'Ġwas', 'ĠTom', 'Â´', 's', 'Ġbirthday', '.']
  clean: ['It', 'Ġwas', 'ĠTom', "'s", 'Ġbirthday', '.']
'Broken � char.'
  raw  : ['Bro', 'ken', 'Ġï¿½', 'Ġchar', '.']
  clean: ['Bro', 'ken', "Ġ'", 'Ġchar', '.']
'She didnt go home.'
  raw  : ['She', 'Ġdidnt', 'Ġgo', 'Ġhome', '.']
  clean: ['She', 'Ġdidn', "'t", 'Ġgo', 'Ġhome', '.']


In [3]:
# texts changed by clean_text
changed = pd.DataFrame(
    [(split, t, clean_text(t)) for split in ds for col in TEXT_COLUMNS for t in ds[split][col] if clean_text(t) != t],
    columns=["split", "raw", "clean"],
)
print(changed["split"].value_counts().to_string())
changed.head(8)

split
train         193
validation     12
test           12


,split,raw,clean
0,train,Jo��o was frustrated at his country's government.,Jo'o was frustrated at his country's government.
1,train,He opened the envelope and was disappointed he...,He opened the envelope and was disappointed he...
2,train,Alex took his clothes to his Mom���s house for...,Alex took his clothes to his Mom's house for h...
3,train,Chloe doesnt want to dance.,Chloe doesn't want to dance.
4,train,"Iris catch down Lisa’s phone to the floor."".","Iris catch down Lisa's phone to the floor.""."
5,train,Jennifer asked her doctor why she wasnt losing...,Jennifer asked her doctor why she wasn't losin...
6,train,John’s airline decided to take its workers on ...,John's airline decided to take its workers on ...
7,train,"OUCH! Mary's foot was hurt, but she kept on p...","OUCH! Mary's foot was hurt, but she kept on pl..."


In [4]:
# subword examples (Ġ = token starts with a space)
for text in ["Kennedy Space Center", "spaceship", "saddened players", "unhappy", "after 20 minutes", "didn't", "Kayla"]:
    print(f"{text!r:24} -> {tok.tokenize(' ' + text)}")

'Kennedy Space Center'   -> ['ĠKennedy', 'ĠSpace', 'ĠCenter']
'spaceship'              -> ['Ġspaceship']
'saddened players'       -> ['Ġsaddened', 'Ġplayers']
'unhappy'                -> ['Ġunhappy']
'after 20 minutes'       -> ['Ġafter', 'Ġ20', 'Ġminutes']
"didn't"                 -> ['Ġdidn', "'t"]
'Kayla'                  -> ['ĠKay', 'la']


In [5]:
# how many train words are a single token
words = Counter(w for col in TEXT_COLUMNS for t in ds["train"][col] for w in re.findall(r"[A-Za-z']+", clean_text(t)))
n_pieces = pd.Series({w: len(tok.tokenize(" " + w)) for w in words})
counts = pd.Series(words)
print(f"distinct words: {len(n_pieces)}")
print(f"single token: {(n_pieces == 1).mean():.1%} of distinct words, {counts[n_pieces == 1].sum() / counts.sum():.1%} of occurrences")
for w in n_pieces.sort_values(ascending=False).index[:8]:
    print(f"  {w:20} {tok.tokenize(' ' + w)}")

distinct words: 18201
single token: 62.0% of distinct words, 95.7% of occurrences
  BOYFRIEND            ['ĠBO', 'Y', 'F', 'RI', 'END']
  unenthusiastic       ['Ġun', 'ent', 'hus', 'i', 'astic']
  acupuncturist        ['Ġac', 'up', 'unct', 'ur', 'ist']
  COURBIN'S            ['ĠCOUR', 'B', 'IN', "'", 'S']
  hypochondriacal      ['Ġhyp', 'ochond', 'ri', 'ac', 'al']
  bioluminescence      ['Ġb', 'iol', 'um', 'ines', 'cence']
  mousetraps           ['Ġm', 'ous', 'et', 'ra', 'ps']
  unceremoniously      ['Ġun', 'ce', 're', 'mon', 'iously']


In [6]:
# input formats for the first train example
encoded = {fmt: {split: encode_split(ds[split], tok, fmt) for split in ds} for fmt in INPUT_FORMATS}
print("label:", ds["train"][0]["label"])
for fmt in INPUT_FORMATS:
    print(fmt)
    for ids in encoded[fmt]["train"][0]["input_ids"]:
        print("  ", tok.decode(ids))

label: 2
separate
   <s>A baby panda was lost in the woods. He was never lost. He was reunited with his parents, who also lived at the zoo.</s>
   <s>A baby panda was lost in the woods. All the people from the zoo looked for him. He was reunited with his parents, who also lived at the zoo.</s>
h_last
   <s>A baby panda was lost in the woods. He was reunited with his parents, who also lived at the zoo.</s></s>He was never lost.</s>
   <s>A baby panda was lost in the woods. He was reunited with his parents, who also lived at the zoo.</s></s>All the people from the zoo looked for him.</s>
h_only
   <s>He was never lost.</s>
   <s>All the people from the zoo looked for him.</s>
joint
   <s>A baby panda was lost in the woods.</s></s>He was never lost.</s></s>All the people from the zoo looked for him.</s></s>He was reunited with his parents, who also lived at the zoo.</s>


In [7]:
# sequence lengths (incl. special tokens), limit is 512
pd.DataFrame({
    (fmt, split): pd.Series([len(ids) for ex in samples for ids in ex["input_ids"]]).describe()[["mean", "50%", "max"]]
    for fmt in INPUT_FORMATS for split, samples in encoded[fmt].items()
}).T.round(1)

mean   50%    max
separate train       32.6  32.0   63.0
         validation  32.9  33.0   58.0
         test        32.8  32.0   60.0
h_last   train       34.8  35.0   65.0
         validation  35.1  35.0   61.0
         test        35.0  35.0   62.0
h_only   train       12.1  12.0   42.0
         validation  12.2  12.0   37.0
         test        12.3  12.0   31.0
joint    train       49.0  48.0  102.0
         validation  49.4  49.0   98.0
         test        49.5  49.0   85.0

In [8]:
# <unk> count and encode -> decode round trip
n_unk = sum(ids.count(tok.unk_token_id) for fmt in encoded.values() for samples in fmt.values()
            for ex in samples for ids in ex["input_ids"])
texts = [clean_text(t) for split in ds for col in TEXT_COLUMNS for t in ds[split][col]]
n_mismatch = sum(tok.decode(tok(t, add_special_tokens=False)["input_ids"]) != t for t in texts)
print(f"<unk>: {n_unk} | round-trip mismatches: {n_mismatch} / {len(texts)}")

<unk>: 0 | round-trip mismatches: 0 / 77308


In [9]:
# token ids per split and column (cleaned text, no special tokens)
ids = {split: {col: tok([clean_text(t) for t in ds[split][col]], add_special_tokens=False)["input_ids"]
               for col in TEXT_COLUMNS} for split in ds}
train_counts = Counter(i for col in TEXT_COLUMNS for seq in ids["train"][col] for i in seq)
print(f"vocab ids seen in train: {len(train_counts)} / {tok.vocab_size} ({len(train_counts) / tok.vocab_size:.1%})")

vocab ids seen in train: 16306 / 50265 (32.4%)


In [10]:
# val/test tokens that never occur in train (embedding stays random in the random-init model)
rows = {}
for split in ["validation", "test"]:
    flat = [i for col in TEXT_COLUMNS for seq in ids[split][col] for i in seq]
    examples = [[i for col in TEXT_COLUMNS for i in ids[split][col][k]] for k in range(len(ds[split]))]
    h_diff = [set(h1) ^ set(h2) for h1, h2 in zip(ids[split]["hypothesis_1"], ids[split]["hypothesis_2"])]
    rows[split] = {
        "unseen token occurrences": np.mean([i not in train_counts for i in flat]),
        "rare (<5x in train) occurrences": np.mean([train_counts[i] < 5 for i in flat]),
        "examples with an unseen token": np.mean([any(i not in train_counts for i in ex) for ex in examples]),
        "examples with an unseen token in h1/h2 difference": np.mean([any(i not in train_counts for i in d) for d in h_diff]),
    }
pd.DataFrame(rows).map(lambda x: f"{x:.1%}")

,validation,test
unseen token occurrences,1.1%,1.1%
rare (<5x in train) occurrences,3.5%,3.4%
examples with an unseen token,26.0%,27.9%
examples with an unseen token in h1/h2 difference,9.1%,14.6%


In [11]:
# most frequent unseen tokens
for split in ["validation", "test"]:
    unseen = Counter(i for col in TEXT_COLUMNS for seq in ids[split][col] for i in seq if i not in train_counts)
    print(split, tok.convert_ids_to_tokens([i for i, _ in unseen.most_common(15)]))

validation ['Yu', 'aris', 'iki', 'oppy', 'Ġespresso', 'Us', 'sa', 'Wilson', 'oshi', 'ĠPass', 'aic', 'oland', 'ĠRoberts', 'Ġchefs', 'ĠRome']
test ['Ġprince', 'iki', 'imi', 'Pri', 'idi', 'anut', 'oun', 'pert', 'ĠMont', 'essor', 'lers', 'ova', 'Ġhikers', 'Cr', 'ĠRafael']


In [12]:
# batch check: shapes, mask = 0 exactly at padding, every sequence starts with <s>
batch = collate(encoded["separate"]["train"][:8], pad_id=tok.pad_token_id)
print({k: tuple(v.shape) for k, v in batch.items()})
assert (batch["attention_mask"] == 0).equal(batch["input_ids"] == tok.pad_token_id)
assert (batch["input_ids"][:, :, 0] == tok.cls_token_id).all()
print("ok")

{'input_ids': (8, 2, 43), 'attention_mask': (8, 2, 43), 'target': (8,)}
ok
